# 🌿 Pineapple Plant Health ML Analysis — Google Colab

**Upload these CSV files when prompted:**
| File | Description |
|------|-------------|
| `plants_ndvi_summary.csv` | Main plant data with NDVI, NDRE, coordinates |
| `plants_predictions_summary.csv` | Predicted growth stage, yield, nitrogen |

**What this notebook does:**
1. Loads and merges your CSV files
2. Engineers 10+ features from NDVI/NDRE
3. Trains XGBoost Regressor (health score 0–100)
4. Trains XGBoost Classifier (Poor / Fair / Good / Excellent)
5. Generates feature importance charts
6. Exports predictions CSV and model PKL files

## Cell 1 — Install packages

In [ ]:
# Run once — installs XGBoost and other required packages
!pip install -q xgboost scikit-learn pandas numpy matplotlib seaborn

## Cell 2 — Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import pickle
import warnings
warnings.filterwarnings('ignore')

from xgboost import XGBRegressor, XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    mean_absolute_error, r2_score,
    accuracy_score, classification_report, confusion_matrix
)
from sklearn.preprocessing import LabelEncoder
from google.colab import files

print('All imports OK')

## Cell 3 — Set file paths

**How to upload your files to Colab runtime storage:**
1. On the left sidebar, click the **folder icon** (Files panel)
2. Click the **upload icon** (paper with up-arrow) at the top of the panel
3. Select both CSV files from your computer:
   - `plants_ndvi_summary.csv`
   - `plants_predictions_summary.csv`
4. Wait until both files appear in the file list (they land at `/content/`)
5. Then run this cell — change the paths below if your filenames differ

In [ ]:
# ── Set paths to your uploaded files ─────────────────────────────────────────
# After uploading via the Files panel, files land at /content/<filename>
# Change these if your filenames are different

NDVI_CSV_PATH = '/content/plants_ndvi_summary.csv'
PRED_CSV_PATH = '/content/plants_predictions_summary.csv'

import os
for p in [NDVI_CSV_PATH, PRED_CSV_PATH]:
    status = '  found' if os.path.exists(p) else '  NOT FOUND — check filename or upload again'
    print(f'{p}{status}')


## Cell 4 — Load and merge data

In [ ]:
# ── Load the two CSVs ──────────────────────────────────────────────────────────
ndvi_df = pd.read_csv(NDVI_CSV_PATH)
pred_df = pd.read_csv(PRED_CSV_PATH)

print(f'NDVI summary  : {ndvi_df.shape[0]:,} rows × {ndvi_df.shape[1]} cols')
print(f'Predictions   : {pred_df.shape[0]:,} rows × {pred_df.shape[1]} cols')

# ── Filter out noise rows ──────────────────────────────────────────────────────
if 'is_noise' in ndvi_df.columns:
    before = len(ndvi_df)
    ndvi_df = ndvi_df[ndvi_df['is_noise'] == False].copy()
    print(f'After noise filter: {len(ndvi_df):,} rows (removed {before - len(ndvi_df):,})')

# ── Merge on plant_id ──────────────────────────────────────────────────────────
df = ndvi_df.merge(pred_df[['plant_id', 'predicted_growth_stage_name',
                              'predicted_yield_kg', 'predicted_nitrogen_status']],
                   on='plant_id', how='left')

print(f'Merged dataset: {df.shape[0]:,} rows × {df.shape[1]} cols')
print('\nColumn list:')
print(df.columns.tolist())


## Cell 5 — Preview data

In [ ]:
print('=== First 5 rows ===')
display(df.head())

print('\n=== NDVI distribution ===')
print(df['ndvi'].describe().round(4))

print('\n=== NDVI category counts ===')
# Use new thresholds if old category names are present
def remap_ndvi_category(v):
    if pd.isna(v): return 'No Data'
    if v < 0.2:   return 'Bare Soil'
    if v < 0.6:   return 'Unhealthy'
    if v < 0.8:   return 'Healthy'
    return 'Saturated'

df['health_class'] = df['ndvi'].apply(remap_ndvi_category)
print(df['health_class'].value_counts())

print('\n=== NDRE availability ===')
ndre_valid = df['ndre'].notna() & (df['ndre'] > -2)
print(f'Valid NDRE: {ndre_valid.sum():,} / {len(df):,} plants')

## Cell 6 — Feature engineering

**Why two feature sets?**
- `health_score` and `health_label` are derived directly from NDVI → using NDVI as a feature causes 100% accuracy (the model just learns the formula, not plant biology)
- Fix: predict health using **NDRE-only features** (a different sensor), so the model must genuinely learn the NDVI–NDRE relationship
- For yield and nitrogen targets, NDVI is a valid input (not leaked)

| Feature | Formula | Used for |
|---------|---------|---------|
| `ndre_filled` | direct | Health + Yield + Nitrogen |
| `nsi` | NDRE − NDVI | Health (NDRE-based), Yield, Nitrogen |
| `chlorophyll_idx` | (NDRE/NDVI) − 1 | Health + Yield + Nitrogen |
| `area_px` | direct | All |
| `row_norm` | row_index / max | All |
| `col_norm` | col_index / max | All |
| `sector_enc` | label-encoded | All |
| `ndvi` | direct | Yield + Nitrogen only |
| `stress_idx` | 1 − NDVI | Yield + Nitrogen only |
| `ndvi_ndre_ratio` | NDVI / (NDRE+1) | Yield + Nitrogen only |
| `growth_stage_enc` | label-encoded | Yield + Nitrogen only |

In [ ]:
# ── Fill missing NDRE with NDVI (best available proxy) ────────────────────────
df['ndre_filled'] = df['ndre'].where(
    df['ndre'].notna() & (df['ndre'] > -2),
    other=df['ndvi']
)

ndvi = df['ndvi'].values
ndre = df['ndre_filled'].values

# ── Derived spectral features ─────────────────────────────────────────────────
df['nsi']             = ndre - ndvi
df['chlorophyll_idx'] = np.where(np.abs(ndvi) > 0.01, (ndre / ndvi) - 1, 0)
df['stress_idx']      = 1 - ndvi
df['ndvi_ndre_ratio'] = ndvi / (ndre + 1 + 1e-6)

# ── Spatial features ──────────────────────────────────────────────────────────
df['row_norm'] = df['row_index'] / (df['row_index'].max() + 1e-6)
df['col_norm'] = df['col_index'] / (df['col_index'].max() + 1e-6)

# ── Categorical encodings ─────────────────────────────────────────────────────
le_sector = LabelEncoder()
df['sector_enc'] = le_sector.fit_transform(df['sector_label'].fillna('UNKNOWN'))

le_stage = LabelEncoder()
df['growth_stage_enc'] = le_stage.fit_transform(
    df['predicted_growth_stage_name'].fillna('Unknown')
)

# ── Feature set A: NDRE-based only (no raw NDVI) — for health models ──────────
# Excludes ndvi, stress_idx, ndvi_ndre_ratio to prevent leakage into health targets
FEATURES_HEALTH = [
    'ndre_filled', 'nsi', 'chlorophyll_idx',
    'area_px', 'row_norm', 'col_norm', 'sector_enc'
]

# ── Feature set B: all features — for yield and nitrogen models ───────────────
FEATURES_ALL = [
    'ndvi', 'ndre_filled', 'nsi', 'chlorophyll_idx',
    'stress_idx', 'ndvi_ndre_ratio', 'area_px',
    'row_norm', 'col_norm', 'sector_enc', 'growth_stage_enc'
]

print(f'FEATURES_HEALTH ({len(FEATURES_HEALTH)}): {FEATURES_HEALTH}')
print(f'FEATURES_ALL    ({len(FEATURES_ALL)}): {FEATURES_ALL}')
print(f'\nAny NaN in FEATURES_HEALTH: {df[FEATURES_HEALTH].isna().any().any()}')
print(f'Any NaN in FEATURES_ALL   : {df[FEATURES_ALL].isna().any().any()}')


## Cell 7 — Build targets and split data

**Three targets — one per model:**

| Model | Target | Features | Type |
|-------|--------|----------|------|
| Health Classifier | NDVI category (Bare Soil / Unhealthy / Healthy / Saturated) | NDRE-only (`FEATURES_HEALTH`) | 4-class classification |
| Yield Regressor | `predicted_yield_kg` | All (`FEATURES_ALL`) | regression |
| Nitrogen Classifier | `predicted_nitrogen_status` (Deficient / Healthy) | All (`FEATURES_ALL`) | binary classification |

In [ ]:
def remap_ndvi_category(v):
    if pd.isna(v): return 'No Data'
    if v < 0.2:   return 'Bare Soil'
    if v < 0.6:   return 'Unhealthy'
    if v < 0.8:   return 'Healthy'
    return 'Saturated'

df['health_class'] = df['ndvi'].apply(remap_ndvi_category)

CATEGORY_MAP = {'Bare Soil': 0, 'Unhealthy': 1, 'Healthy': 2, 'Saturated': 3}
CLASS_NAMES  = ['Bare Soil', 'Unhealthy', 'Healthy', 'Saturated']

# ── Target 1: health category (4-class) ──────────────────────────────────────
df['health_label'] = df['health_class'].map(CATEGORY_MAP)

# ── Target 2: yield in kg ─────────────────────────────────────────────────────
df['yield_kg'] = pd.to_numeric(df['predicted_yield_kg'], errors='coerce')

# ── Target 3: nitrogen binary (0=Healthy, 1=Deficient) ───────────────────────
df['nitrogen_binary'] = (df['predicted_nitrogen_status'] == 'Deficient').astype(int)

# ── Build clean working sets ──────────────────────────────────────────────────
clean_health = df[FEATURES_HEALTH + ['health_label']].dropna()
clean_yield  = df[FEATURES_ALL + ['yield_kg']].dropna()
clean_nitro  = df[FEATURES_ALL + ['nitrogen_binary']].dropna()

print(f'Health classifier rows : {len(clean_health):,}')
print(f'Yield regressor rows   : {len(clean_yield):,}')
print(f'Nitrogen classifier rows: {len(clean_nitro):,}')

# ── Splits ────────────────────────────────────────────────────────────────────
Xh = clean_health[FEATURES_HEALTH].values
yh = clean_health['health_label'].values.astype(int)
Xh_tr, Xh_te, yh_tr, yh_te = train_test_split(Xh, yh, test_size=0.2, random_state=42, stratify=yh)

Xy = clean_yield[FEATURES_ALL].values
yy = clean_yield['yield_kg'].values
Xy_tr, Xy_te, yy_tr, yy_te = train_test_split(Xy, yy, test_size=0.2, random_state=42)

Xn = clean_nitro[FEATURES_ALL].values
yn = clean_nitro['nitrogen_binary'].values
Xn_tr, Xn_te, yn_tr, yn_te = train_test_split(Xn, yn, test_size=0.2, random_state=42, stratify=yn)

print(f'\nHealth  train/test: {len(Xh_tr):,} / {len(Xh_te):,}')
print(f'Yield   train/test: {len(Xy_tr):,} / {len(Xy_te):,}')
print(f'Nitrogen train/test: {len(Xn_tr):,} / {len(Xn_te):,}')

# ── Category distribution chart ───────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
cat_colors = ['#f87171', '#fb923c', '#4ade80', '#15803d']

counts = [np.sum(yh == i) for i in range(4)]
axes[0].bar(CLASS_NAMES, counts, color=cat_colors, edgecolor='white')
axes[0].set_title('Health Category Distribution')
axes[0].set_ylabel('Plant count')
for i, c in enumerate(counts):
    axes[0].text(i, c + 20, f'{c:,}', ha='center', fontsize=9)

axes[1].hist(yy, bins=50, color='#60a5fa', edgecolor='white', linewidth=0.3)
axes[1].set_title('Yield Distribution (kg)')
axes[1].set_xlabel('Predicted yield (kg)')

nitro_counts = [np.sum(yn == 0), np.sum(yn == 1)]
axes[2].bar(['Healthy', 'Deficient'], nitro_counts, color=['#4ade80', '#f87171'], edgecolor='white')
axes[2].set_title('Nitrogen Status Distribution')
for i, c in enumerate(nitro_counts):
    axes[2].text(i, c + 20, f'{c:,}', ha='center', fontsize=9)

plt.tight_layout()
plt.savefig('target_distributions.png', dpi=150, bbox_inches='tight')
plt.show()


## Cell 8 — Model 1: Health Category Classifier (NDRE features → 4 categories)

Uses only NDRE-based features — no raw NDVI — so the model must learn the spectral relationship between NDRE and NDVI-derived health categories. A realistic accuracy here is 70–90%.

In [ ]:
print('Training Health Classifier (NDRE features only)...')

clf_health = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=5,
    reg_alpha=0.1,
    reg_lambda=1.0,
    eval_metric='mlogloss',
    random_state=42,
    n_jobs=-1,
    verbosity=0
)

clf_health.fit(Xh_tr, yh_tr, eval_set=[(Xh_te, yh_te)], verbose=50)

yh_pred = clf_health.predict(Xh_te)
acc_h   = accuracy_score(yh_te, yh_pred)
print(f'\n  Accuracy: {acc_h:.4f} ({acc_h*100:.1f}%)')
print('\nClassification Report:')
print(classification_report(yh_te, yh_pred, target_names=CLASS_NAMES))

cm_h = confusion_matrix(yh_te, yh_pred)
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm_h, annot=True, fmt='d', cmap='Greens',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax)
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
ax.set_title(f'Health Classifier — Confusion Matrix  (Acc={acc_h:.3f})')
plt.tight_layout()
plt.savefig('confusion_matrix_health.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: confusion_matrix_health.png')


## Cell 9 — Model 2: Yield Regressor (all features → kg per plant)

In [ ]:
print('Training Yield Regressor...')

reg_yield = XGBRegressor(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=5,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=42,
    n_jobs=-1,
    verbosity=0
)

reg_yield.fit(Xy_tr, yy_tr, eval_set=[(Xy_te, yy_te)], verbose=50)

yy_pred = reg_yield.predict(Xy_te)
mae_y   = mean_absolute_error(yy_te, yy_pred)
r2_y    = r2_score(yy_te, yy_pred)
print(f'\n  MAE : {mae_y:.4f} kg')
print(f'  R²  : {r2_y:.4f}')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].scatter(yy_te, yy_pred, alpha=0.3, s=8, color='#60a5fa')
mn, mx = yy_te.min(), yy_te.max()
axes[0].plot([mn, mx], [mn, mx], 'r--', linewidth=1)
axes[0].set_xlabel('Actual yield (kg)')
axes[0].set_ylabel('Predicted yield (kg)')
axes[0].set_title(f'Yield: Predicted vs Actual  (R²={r2_y:.3f})')

res = yy_pred - yy_te
axes[1].hist(res, bins=60, color='#60a5fa', edgecolor='white', linewidth=0.3)
axes[1].axvline(0, color='red', linestyle='--', linewidth=1)
axes[1].set_xlabel('Residual (kg)')
axes[1].set_title(f'Yield Residuals  (MAE={mae_y:.4f} kg)')

plt.tight_layout()
plt.savefig('regression_yield.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: regression_yield.png')


## Cell 9b — Model 3: Nitrogen Deficiency Classifier (all features → Deficient / Healthy)

In [ ]:
print('Training Nitrogen Deficiency Classifier...')

# Class weight to handle imbalance (Deficient plants are fewer)
deficient_ratio = np.sum(yn_tr == 0) / (np.sum(yn_tr == 1) + 1e-6)

clf_nitro = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=5,
    reg_alpha=0.1,
    reg_lambda=1.0,
    scale_pos_weight=deficient_ratio,
    eval_metric='logloss',
    random_state=42,
    n_jobs=-1,
    verbosity=0
)

clf_nitro.fit(Xn_tr, yn_tr, eval_set=[(Xn_te, yn_te)], verbose=50)

yn_pred  = clf_nitro.predict(Xn_te)
acc_n    = accuracy_score(yn_te, yn_pred)
print(f'\n  Accuracy: {acc_n:.4f} ({acc_n*100:.1f}%)')
print('\nClassification Report:')
print(classification_report(yn_te, yn_pred, target_names=['Healthy', 'Deficient']))

cm_n = confusion_matrix(yn_te, yn_pred)
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm_n, annot=True, fmt='d', cmap='Oranges',
            xticklabels=['Healthy', 'Deficient'],
            yticklabels=['Healthy', 'Deficient'], ax=ax)
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
ax.set_title(f'Nitrogen Classifier — Confusion Matrix  (Acc={acc_n:.3f})')
plt.tight_layout()
plt.savefig('confusion_matrix_nitrogen.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: confusion_matrix_nitrogen.png')


## Cell 10 — Feature importance

In [ ]:
fi_health = pd.Series(clf_health.feature_importances_, index=FEATURES_HEALTH).sort_values(ascending=True)
fi_yield  = pd.Series(reg_yield.feature_importances_,  index=FEATURES_ALL).sort_values(ascending=True)
fi_nitro  = pd.Series(clf_nitro.feature_importances_,  index=FEATURES_ALL).sort_values(ascending=True)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

fi_health.plot(kind='barh', ax=axes[0], color='#4ade80', edgecolor='white')
axes[0].set_title('Health Classifier\n(NDRE features → category)')
axes[0].set_xlabel('Importance')

fi_yield.plot(kind='barh', ax=axes[1], color='#60a5fa', edgecolor='white')
axes[1].set_title('Yield Regressor\n(all features → kg)')
axes[1].set_xlabel('Importance')

fi_nitro.plot(kind='barh', ax=axes[2], color='#fb923c', edgecolor='white')
axes[2].set_title('Nitrogen Classifier\n(all features → Deficient/Healthy)')
axes[2].set_xlabel('Importance')

plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

fi_df = pd.DataFrame({
    'feature_health':   FEATURES_HEALTH + ['-'] * (len(FEATURES_ALL) - len(FEATURES_HEALTH)),
    'importance_health': list(clf_health.feature_importances_) + [None] * (len(FEATURES_ALL) - len(FEATURES_HEALTH)),
    'feature_all':      FEATURES_ALL,
    'importance_yield': reg_yield.feature_importances_,
    'importance_nitro': clf_nitro.feature_importances_,
})
fi_df.to_csv('feature_importance.csv', index=False)
print('Saved: feature_importance.png')
print('Saved: feature_importance.csv')
display(fi_df)


## Cell 11 — Predict on all plants

In [ ]:
CLASS_NAMES = ['Bare Soil', 'Unhealthy', 'Healthy', 'Saturated']

REC = {
    'Bare Soil':  'URGENT: Replanting required. Check soil nutrients and irrigation.',
    'Unhealthy':  'WARNING: Immediate field scouting. Apply nitrogen/potassium treatment.',
    'Healthy':    'GOOD: Maintain current protocols. Monitor weekly.',
    'Saturated':  'INFO: Peak canopy. Use NDRE for maturity tracking. Plan harvest window.',
}

# ── Health predictions (NDRE features only) ───────────────────────────────────
Xh_all = df[FEATURES_HEALTH].dropna()
idx_h  = Xh_all.index
df.loc[idx_h, 'pred_health_cat']  = [CLASS_NAMES[c] for c in clf_health.predict(Xh_all.values)]
df.loc[idx_h, 'recommendation']   = df.loc[idx_h, 'pred_health_cat'].map(REC)

# ── Yield predictions (all features) ─────────────────────────────────────────
Xy_all = df[FEATURES_ALL].dropna()
idx_y  = Xy_all.index
df.loc[idx_y, 'pred_yield_kg']    = reg_yield.predict(Xy_all.values)

# ── Nitrogen predictions (all features) ───────────────────────────────────────
df.loc[idx_y, 'pred_nitrogen']    = ['Deficient' if p == 1 else 'Healthy'
                                      for p in clf_nitro.predict(Xy_all.values)]

print('Health category breakdown (from NDRE features):')
print(df['pred_health_cat'].value_counts())
print(f'\nMean predicted yield : {df["pred_yield_kg"].mean():.3f} kg/plant')
print(f'Nitrogen Deficient   : {(df["pred_nitrogen"]=="Deficient").sum():,} plants')
print(f'Nitrogen Healthy     : {(df["pred_nitrogen"]=="Healthy").sum():,} plants')


## Cell 12 — Sector analysis

In [ ]:
sector_stats = df.groupby('sector_label').agg(
    plant_count   = ('plant_id',         'count'),
    avg_ndvi      = ('ndvi',              'mean'),
    avg_ndre      = ('ndre_filled',       'mean'),
    avg_yield     = ('pred_yield_kg',     'mean'),
    pct_bare      = ('pred_health_cat',   lambda x: (x=='Bare Soil').mean() * 100),
    pct_unhealthy = ('pred_health_cat',   lambda x: (x=='Unhealthy').mean() * 100),
    pct_healthy   = ('pred_health_cat',   lambda x: (x=='Healthy').mean() * 100),
    pct_saturated = ('pred_health_cat',   lambda x: (x=='Saturated').mean() * 100),
    pct_deficient = ('pred_nitrogen',     lambda x: (x=='Deficient').mean() * 100),
).round(2).reset_index()

# Priority flag: sectors with >20% bare soil or >30% unhealthy
sector_stats['priority'] = np.where(
    (sector_stats['pct_bare'] > 20) | (sector_stats['pct_unhealthy'] > 30),
    'HIGH', 'NORMAL'
)

# Cost estimates (INR)
PIXEL_AREA_M2 = 0.05 * 0.05
PIXEL_AREA_HA = PIXEL_AREA_M2 / 10000
COST_PER_HA   = {'bare': 5000, 'unhealthy': 8000, 'healthy': 3000}

avg_area_px = df['area_px'].median() if 'area_px' in df.columns else 500
sector_stats['est_area_ha']  = sector_stats['plant_count'] * avg_area_px * PIXEL_AREA_HA
sector_stats['est_cost_inr'] = (
    sector_stats['pct_bare']      / 100 * sector_stats['est_area_ha'] * COST_PER_HA['bare']      +
    sector_stats['pct_unhealthy'] / 100 * sector_stats['est_area_ha'] * COST_PER_HA['unhealthy'] +
    sector_stats['pct_healthy']   / 100 * sector_stats['est_area_ha'] * COST_PER_HA['healthy']
).astype(int)

sector_stats.to_csv('sector_management_plan.csv', index=False)
print('Saved: sector_management_plan.csv')
print(f'Total estimated treatment cost: ₹{sector_stats["est_cost_inr"].sum():,}')
print(f'HIGH priority sectors: {(sector_stats["priority"]=="HIGH").sum()}')
display(sector_stats.sort_values('pct_bare', ascending=False).head(10))


## Cell 13 — Farm health report

In [ ]:
total   = len(df.dropna(subset=['pred_health_cat']))
counts  = df['pred_health_cat'].value_counts()
n_bare  = counts.get('Bare Soil',  0)
n_unhl  = counts.get('Unhealthy',  0)
n_hlthy = counts.get('Healthy',    0)
n_sat   = counts.get('Saturated',  0)
avg_yld = df['pred_yield_kg'].mean()
n_def   = (df['pred_nitrogen'] == 'Deficient').sum()

# Grade based on % healthy+saturated
pct_good = (n_hlthy + n_sat) / total * 100
grade = ('A' if pct_good>=80 else 'B' if pct_good>=65 else 'C' if pct_good>=50
         else 'D' if pct_good>=35 else 'F')

report = f"""
╔══════════════════════════════════════════════════════════╗
║           PINEAPPLE FARM HEALTH REPORT                  ║
╠══════════════════════════════════════════════════════════╣
║  Total plants analysed : {total:>8,}                    ║
║  Mean predicted yield  : {avg_yld:>7.3f} kg / plant          ║
║  Nitrogen deficient    : {n_def:>8,} plants              ║
║  Farm health grade     : {grade:>8}                    ║
╠══════════════════════════════════════════════════════════╣
║  CATEGORY BREAKDOWN                                      ║
║  Saturated  (≥0.8 NDVI) : {n_sat:>6,}  ({n_sat/total*100:5.1f}%)           ║
║  Healthy  (0.6–0.8 NDVI): {n_hlthy:>6,}  ({n_hlthy/total*100:5.1f}%)           ║
║  Unhealthy (0.2–0.6 NDVI): {n_unhl:>5,}  ({n_unhl/total*100:5.1f}%)           ║
║  Bare Soil  (<0.2 NDVI)  : {n_bare:>5,}  ({n_bare/total*100:5.1f}%)           ║
╠══════════════════════════════════════════════════════════╣
║  ACTIONS REQUIRED                                        ║"""

if n_bare > 0:
    report += f"\n║  URGENT : {n_bare:,} plants need replanting (Bare Soil)        ║"
if n_unhl > 0:
    report += f"\n║  WARNING: {n_unhl:,} plants need treatment (Unhealthy)         ║"
if n_def > 0:
    report += f"\n║  NITRO  : {n_def:,} plants show nitrogen deficiency           ║"
if n_hlthy > 0:
    report += f"\n║  GOOD   : {n_hlthy:,} plants on track — maintain protocols    ║"
if n_sat > 0:
    report += f"\n║  HARVEST: {n_sat:,} plants at peak — monitor harvest window  ║"

report += "\n╚══════════════════════════════════════════════════════════╝"

print(report)
with open('plant_health_report.txt', 'w') as f:
    f.write(report)
print('Saved: plant_health_report.txt')


## Cell 14 — Summary chart

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('Pineapple Farm Health Summary', fontsize=14, fontweight='bold')

cat_colors = ['#f87171', '#fb923c', '#4ade80', '#15803d']

# 1. Health category bar
ax = axes[0, 0]
vals = [n_bare, n_unhl, n_hlthy, n_sat]
bars = ax.bar(CLASS_NAMES, vals, color=cat_colors, edgecolor='white')
ax.set_title('Health Category Counts')
ax.set_ylabel('Plants')
for b, v in zip(bars, vals):
    ax.text(b.get_x() + b.get_width()/2, b.get_height() + 20, f'{v:,}',
            ha='center', fontsize=8)

# 2. Health pie
ax = axes[0, 1]
ax.pie(vals, labels=CLASS_NAMES, colors=cat_colors, autopct='%1.1f%%',
       startangle=90, pctdistance=0.8)
ax.set_title('Category Distribution')

# 3. NDVI histogram
ax = axes[0, 2]
ndvi_vals = df['ndvi'].dropna().values
n, bins, patches = ax.hist(ndvi_vals, bins=50, edgecolor='white', linewidth=0.3)
for patch, left in zip(patches, bins[:-1]):
    if left < 0.2:   patch.set_facecolor('#f87171')
    elif left < 0.6: patch.set_facecolor('#fb923c')
    elif left < 0.8: patch.set_facecolor('#4ade80')
    else:            patch.set_facecolor('#15803d')
for v in [0.2, 0.6, 0.8]:
    ax.axvline(v, color='gray', linestyle='--', linewidth=0.8)
ax.set_title('NDVI Distribution')
ax.set_xlabel('NDVI value')

# 4. Yield distribution
ax = axes[1, 0]
yld_vals = df['pred_yield_kg'].dropna().values
ax.hist(yld_vals, bins=50, color='#60a5fa', edgecolor='white', linewidth=0.3)
ax.set_title('Predicted Yield Distribution')
ax.set_xlabel('Yield (kg/plant)')

# 5. Sector avg yield (top 15)
ax = axes[1, 1]
top_sectors = sector_stats.nlargest(15, 'avg_yield')
bar_colors  = ['#f87171' if p == 'HIGH' else '#4ade80' for p in top_sectors['priority']]
ax.barh(top_sectors['sector_label'], top_sectors['avg_yield'],
        color=bar_colors, edgecolor='white')
ax.set_title('Avg Yield by Sector (top 15)\nRed = HIGH priority')
ax.set_xlabel('Avg yield (kg)')

# 6. Feature importance (health classifier)
ax = axes[1, 2]
fi_health.tail(7).plot(kind='barh', ax=ax, color='#4ade80', edgecolor='white')
ax.set_title('Health Classifier\nFeature Importances')
ax.set_xlabel('Importance')

plt.tight_layout()
plt.savefig('plant_health_report.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: plant_health_report.png')


## Cell 15 — Save all outputs and download

In [ ]:
# ── Save all three model PKL files ───────────────────────────────────────────
with open('xgb_health_classifier.pkl', 'wb') as f:
    pickle.dump(clf_health, f)
with open('xgb_yield_regressor.pkl', 'wb') as f:
    pickle.dump(reg_yield, f)
with open('xgb_nitrogen_classifier.pkl', 'wb') as f:
    pickle.dump(clf_nitro, f)
with open('label_encoders.pkl', 'wb') as f:
    pickle.dump({'sector': le_sector, 'stage': le_stage}, f)

print('Saved: xgb_health_classifier.pkl')
print('Saved: xgb_yield_regressor.pkl')
print('Saved: xgb_nitrogen_classifier.pkl')
print('Saved: label_encoders.pkl')

# ── Save predictions CSV ──────────────────────────────────────────────────────
output_cols = [
    'plant_id', 'sector_label', 'lat', 'lon',
    'ndvi', 'ndre_filled',
    'health_class', 'pred_health_cat', 'recommendation',
    'pred_yield_kg', 'pred_nitrogen'
]
out_df = df[[c for c in output_cols if c in df.columns]].copy()
out_df.to_csv('plant_health_predictions.csv', index=False)
print(f'Saved: plant_health_predictions.csv  ({len(out_df):,} rows)')

# ── Download all files ────────────────────────────────────────────────────────
print('\nDownloading all output files...')
for fname in [
    'plant_health_predictions.csv',
    'sector_management_plan.csv',
    'feature_importance.csv',
    'plant_health_report.txt',
    'plant_health_report.png',
    'feature_importance.png',
    'confusion_matrix_health.png',
    'confusion_matrix_nitrogen.png',
    'regression_yield.png',
    'target_distributions.png',
    'xgb_health_classifier.pkl',
    'xgb_yield_regressor.pkl',
    'xgb_nitrogen_classifier.pkl',
    'label_encoders.pkl',
]:
    try:
        files.download(fname)
        print(f'  ✓ {fname}')
    except Exception as e:
        print(f'  ✗ {fname}: {e}')


## Done!

### Output files created
| File | Description |
|------|-------------|
| `plant_health_predictions.csv` | All plants with predicted score + category + recommendation |
| `sector_management_plan.csv` | Per-sector health stats and INR cost estimate |
| `feature_importance.csv` | Feature importance from both models |
| `plant_health_report.txt` | Text summary report |
| `plant_health_report.png` | 6-panel summary chart |
| `xgb_health_regressor.pkl` | Trained regression model |
| `xgb_health_classifier.pkl` | Trained classification model |
| `label_encoders.pkl` | Sector + stage encoders for inference |